# Complete offshore oil-and-gas process engineering study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/equinor/neqsim/blob/master/examples/notebooks/complete_offshore_process_engineering_study.ipynb)

This notebook turns the full `comparesimulations.ipynb` offshore process into a governed engineering study. It retains the published PR-EOS fluid, three-stage oil stabilization, LP/MP recompression, two-stage export compression, low-temperature liquid recovery, fuel-gas takeoff, oil export pumping, and recycle. It then executes:

1. the published normal operating point and a numerical benchmark;
2. a controlled operating-envelope case matrix;
3. the closed process-to-engineering design loop;
4. equipment, piping, valve, instrument, safety, materials, and preliminary mechanical calculations;
5. coordinated registers, datasheets, DEXPI 2.0, qualification gaps, and an explicit approval boundary.

```text
well fluid -> 20-VA-01 -> 20-VA-02 -> 20-VA-03 -> oil cooler/pump -> export oil
                 |           |           |
                 +---- LP/MP recompression ----+-> 23-KA-01 -> LTS -> 27-KA-01 -> export gas
                                               |       ^
                                               + fuel  + condensate recycle
```

The study is suitable for concept and pre-FEED screening. Synthetic screening assumptions are never represented as HAZOP, vendor, code, or construction approval; `fitnessForConstruction` remains `false`.


In [ ]:
import json, os, shutil, sys
from pathlib import Path

def find_neqsim_project_root():
    candidates = []
    if os.environ.get('NEQSIM_PROJECT_ROOT'):
        candidates.append(Path(os.environ['NEQSIM_PROJECT_ROOT']).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / 'pom.xml').exists() and (candidate / 'devtools/neqsim_dev_setup.py').exists():
            return candidate
    raise RuntimeError('Run from a NeqSim checkout or set NEQSIM_PROJECT_ROOT')

ROOT = find_neqsim_project_root()
sys.path.insert(0, str(ROOT / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(project_root=ROOT,
    recompile=not (ROOT / 'target/classes').exists(), verbose=False))

import jpype
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

JClass, JProxy = ns.JClass, jpype.JProxy
jdouble = lambda values: jpype.JArray(jpype.JDouble)(values)
print('Loaded NeqSim workspace classes from', ROOT)


## 1. Controlled process and design basis

The composition and normal operating point below are transcribed from the referenced Colab notebook. Pressure inputs use gauge pressure where the source does; process equipment reports absolute pressure. Added gas and oil export-line objects have negligible base-case pressure drop and allow the hydraulic design modules to size real terminal lines.


In [ ]:
BASE = dict(
    feed_rate_kmol_h=8000.0, feed_temperature_C=60.0,
    Psep1_barg=31.5, Tsep1_C=70.0, Psep2_barg=8.0, Tsep2_C=68.2,
    Psep3_barg=1.5, Tsep3_C=65.0, scrubber_temperature_C=32.0,
    export_interstage_barg=90.0, refrigeration_C=10.0,
    oil_export_barg=60.0, oil_export_C=48.5,
    gas_export_barg=188.6, gas_export_C=40.0,
)

PUBLISHED = {
    'Gas export rate [kmol/h]': 5111.623526578303,
    'Oil export rate [kmol/h]': 2753.3527498934595,
    'Oil molecular weight [g/mol]': 202.29750035522014,
    'Gas molecular weight [g/mol]': 21.055380161191643,
    'Oil RVP [psia]': 10.254893444029019,
    'Oil TVP [bara]': 1.7263686152854913,
    'Gas GCV 15/15 [MJ/Sm3]': 45.67781731945272,
    'Gas Wobbe 15/15 [MJ/Sm3]': 53.49061269952718,
    'Total rotating power [kW]': 10376.125483501624,
}

COMPONENTS = [
    ('CO2', 1.5870), ('methane', 52.51), ('ethane', 6.24),
    ('propane', 4.23), ('i-butane', 0.855), ('n-butane', 2.213),
    ('i-pentane', 1.124), ('n-pentane', 1.271), ('n-hexane', 2.289),
]
TBP_CUTS = [
    ('C7+_cut1', 0.8501, 108.47, 0.7411), ('C7+_cut2', 1.2802, 120.40, 0.7550),
    ('C7+_cut3', 1.6603, 133.64, 0.7695), ('C7+_cut4', 6.5311, 164.70, 0.7990),
    ('C7+_cut5', 6.3311, 215.94, 0.8387), ('C7+_cut6', 4.9618, 273.34, 0.8754),
    ('C7+_cut7', 2.9105, 334.92, 0.90731), ('C7+_cut8', 3.0505, 412.79, 0.94575),
]
pd.DataFrame([BASE]).T.rename(columns={0: 'normal operating point'})


In [ ]:
SystemPrEos = JClass('neqsim.thermo.system.SystemPrEos')
Stream = JClass('neqsim.process.equipment.stream.Stream')
ThreePhaseSeparator = JClass('neqsim.process.equipment.separator.ThreePhaseSeparator')
Separator = JClass('neqsim.process.equipment.separator.Separator')
Heater = JClass('neqsim.process.equipment.heatexchanger.Heater')
Cooler = JClass('neqsim.process.equipment.heatexchanger.Cooler')
HeatExchanger = JClass('neqsim.process.equipment.heatexchanger.HeatExchanger')
Compressor = JClass('neqsim.process.equipment.compressor.Compressor')
Pump = JClass('neqsim.process.equipment.pump.Pump')
ThrottlingValve = JClass('neqsim.process.equipment.valve.ThrottlingValve')
Mixer = JClass('neqsim.process.equipment.mixer.Mixer')
Splitter = JClass('neqsim.process.equipment.splitter.Splitter')
Recycle = JClass('neqsim.process.equipment.util.Recycle')
AdiabaticPipe = JClass('neqsim.process.equipment.pipeline.AdiabaticPipe')
ProcessSystem = JClass('neqsim.process.processmodel.ProcessSystem')

def build_fluid():
    fluid = SystemPrEos()
    fluid.getCharacterization().setTBPModel('Two')
    for name, amount in COMPONENTS:
        fluid.addComponent(name, amount)
    for name, amount, molecular_weight_g_mol, density in TBP_CUTS:
        fluid.addTBPfraction(name, amount, molecular_weight_g_mol / 1000.0, density)
    fluid.setMixingRule('classic')
    return fluid

def build_process():
    wellstream = Stream('well stream', build_fluid())
    wellstream.setTemperature(BASE['feed_temperature_C'], 'C')
    wellstream.setPressure(BASE['Psep1_barg'] + 0.5, 'barg')

    h20_01 = Heater('20-HA-01', wellstream)
    sep20_01 = ThreePhaseSeparator('20-VA-01', h20_01.getOutStream())
    vlv100 = ThrottlingValve('VLV-100', sep20_01.getOilOutStream())
    vlv100.setCv(800.0); vlv100.setPercentValveOpening(70.0)
    mix101 = Mixer('MIX-101'); mix101.addStream(vlv100.getOutStream())
    h20_02 = Heater('20-HA-02', mix101.getOutStream())
    sep20_02 = ThreePhaseSeparator('20-VA-02', h20_02.getOutStream())
    vlv102 = ThrottlingValve('VLV-102', sep20_02.getOilOutStream())
    vlv102.setCv(600.0); vlv102.setPercentValveOpening(70.0)

    oil_reflux = wellstream.clone(); oil_reflux.setName('third stage reflux')
    oil_reflux.setFlowRate(1.0e-6, 'kg/hr')
    mix102 = Mixer('MIX-102'); mix102.addStream(vlv102.getOutStream()); mix102.addStream(oil_reflux)
    h20_03 = Heater('20-HA-03', mix102.getOutletStream())
    sep20_03 = ThreePhaseSeparator('20-VA-03', h20_03.getOutStream())

    h23_03 = Cooler('23-HA-03', sep20_03.getGasOutStream())
    scrub23_03 = Separator('23-VG-03', h23_03.getOutStream())
    p23_suction = AdiabaticPipe('23-PA-01-SUCTION', scrub23_03.getLiquidOutStream())
    p23_suction.setLength(20.0); p23_suction.setDiameter(0.154)
    p23_suction.setPipeWallRoughness(4.6e-5)
    p23_suction.setInletElevation(6.0); p23_suction.setOutletElevation(0.0)
    p23_01 = Pump('23-PA-01', p23_suction.getOutStream())
    lp_recycle = Recycle('LP oil recycle'); lp_recycle.addStream(p23_01.getOutStream())
    lp_recycle.setOutletStream(oil_reflux); lp_recycle.setTolerance(1.0e-6)
    k23_03 = Compressor('23-KA-03', scrub23_03.getGasOutStream())
    k23_03.setIsentropicEfficiency(0.75)

    mix103 = Mixer('MIX-103'); mix103.addStream(k23_03.getOutStream())
    mix103.addStream(sep20_02.getGasOutStream())
    h23_02 = Cooler('23-HA-02', mix103.getOutStream())
    scrub23_02 = Separator('23-VG-02', h23_02.getOutStream())
    mix102.addStream(scrub23_02.getLiquidOutStream())
    k23_02 = Compressor('23-KA-02', scrub23_02.getGasOutStream())
    k23_02.setIsentropicEfficiency(0.75)

    mix100 = Mixer('MIX-100'); mix100.addStream(k23_02.getOutStream())
    mix100.addStream(sep20_01.getGasOutStream())
    h23_01 = Cooler('23-HA-01', mix100.getOutStream())
    scrub23_01 = Separator('23-VG-01', h23_01.getOutStream())
    mix101.addStream(scrub23_01.getLiquidOutStream())
    k23_01 = Compressor('23-KA-01', scrub23_01.getGasOutStream())
    k23_01.setIsentropicEfficiency(0.75)

    h24_01 = Cooler('24-HA-01', k23_01.getOutStream())
    scrub24_01 = Separator('24-VG-01', h24_01.getOutStream())
    mix101.addStream(scrub24_01.getLiquidOutStream())
    gas_splitter = Splitter('fuel gas splitter', scrub24_01.getGasOutStream())
    gas_splitter.setSplitNumber(2); gas_splitter.setFlowRates(jdouble([-1.0, 2966.0]), 'kg/hr')
    fuel_gas = gas_splitter.getSplitStream(1); fuel_gas.setName('fuel gas')

    h25_01 = HeatExchanger('25-HA-01', gas_splitter.getSplitStream(0))
    h25_01.setGuessOutTemperature(288.15); h25_01.setUAvalue(800.0e3)
    h25_02 = Cooler('25-HA-02', h25_01.getOutStream(0))
    scrub25_01 = Separator('25-VG-01', h25_02.getOutStream())
    mix100.addStream(scrub25_01.getLiquidOutStream())
    h25_01.setFeedStream(1, scrub25_01.getGasOutStream())
    k27_01 = Compressor('27-KA-01', h25_01.getOutStream(1))
    k27_01.setIsentropicEfficiency(0.75)
    h27_01 = Cooler('27-HA-01', k27_01.getOutStream())
    export_gas = h27_01.getOutStream(); export_gas.setName('export gas')
    gas_line = AdiabaticPipe('EXPORT-GAS-LINE', export_gas)
    gas_line.setLength(2000.0); gas_line.setDiameter(0.4064); gas_line.setPipeWallRoughness(4.6e-5)

    h21_01 = Cooler('21-HA-01', sep20_03.getOilOutStream())
    p21_suction = AdiabaticPipe('21-PA-01-SUCTION', h21_01.getOutStream())
    p21_suction.setLength(30.0); p21_suction.setDiameter(0.203)
    p21_suction.setPipeWallRoughness(4.6e-5)
    p21_suction.setInletElevation(15.0); p21_suction.setOutletElevation(0.0)
    p21_01 = Pump('21-PA-01', p21_suction.getOutStream())
    export_oil = p21_01.getOutStream(); export_oil.setName('export oil')
    oil_line = AdiabaticPipe('EXPORT-OIL-LINE', export_oil)
    oil_line.setLength(2000.0); oil_line.setDiameter(0.4064); oil_line.setPipeWallRoughness(4.6e-5)

    process = ProcessSystem('Complete offshore oil and gas process')
    for unit in [wellstream, h20_01, sep20_01, vlv100, mix101, h20_02, sep20_02,
                 vlv102, oil_reflux, mix102, h20_03, sep20_03, h23_03, scrub23_03,
                 p23_suction, p23_01, lp_recycle, k23_03, mix103, h23_02, scrub23_02, k23_02,
                 mix100, h23_01, scrub23_01, k23_01, h24_01, scrub24_01, gas_splitter,
                 h25_01, h25_02, scrub25_01, k27_01, h27_01, h21_01, p21_suction, p21_01,
                 export_gas, export_oil, fuel_gas, gas_line, oil_line]:
        process.add(unit)
    apply_case(process, BASE)
    return process

def apply_case(process, values):
    p1, p2, p3 = values['Psep1_barg'], values['Psep2_barg'], values['Psep3_barg']
    feed = process.getUnit('well stream')
    feed.setFlowRate(values['feed_rate_kmol_h'] * 1000.0 / 3600.0, 'mol/sec')
    feed.setTemperature(values['feed_temperature_C'], 'C'); feed.setPressure(p1 + 0.5, 'barg')
    settings = {
        '20-HA-01': (values['Tsep1_C'], p1), '20-HA-02': (values['Tsep2_C'], p2),
        '20-HA-03': (values['Tsep3_C'], p3),
        '23-HA-03': (values['scrubber_temperature_C'], p3 - 0.5),
        '23-HA-02': (values['scrubber_temperature_C'], p2 - 1.0),
        '23-HA-01': (values['scrubber_temperature_C'], p1 - 0.3),
        '24-HA-01': (30.0, values['export_interstage_barg'] - 1.0),
        '25-HA-02': (values['refrigeration_C'], values['export_interstage_barg'] - 1.5),
        '21-HA-01': (values['oil_export_C'], p3),
        '27-HA-01': (values['gas_export_C'], values['gas_export_barg']),
    }
    for tag, (temperature, pressure) in settings.items():
        unit = process.getUnit(tag); unit.setOutTemperature(temperature, 'C'); unit.setOutPressure(pressure, 'barg')
    process.getUnit('VLV-100').setOutletPressure(p2 + 0.5, 'barg')
    process.getUnit('VLV-102').setOutletPressure(p3 + 0.5, 'barg')
    process.getUnit('23-PA-01').setOutletPressure(p3 + 0.5, 'barg')
    process.getUnit('23-KA-03').setOutletPressure(p2, 'barg')
    process.getUnit('23-KA-02').setOutletPressure(p1, 'barg')
    process.getUnit('23-KA-01').setOutletPressure(values['export_interstage_barg'], 'barg')
    process.getUnit('27-KA-01').setOutletPressure(values['gas_export_barg'], 'barg')
    process.getUnit('21-PA-01').setOutletPressure(values['oil_export_barg'] + 1.01325, 'bara')


## 2. Normal-case simulation and benchmark

The benchmark checks material balance, both product rates and qualities, all major duties, rotating power, and discharge temperature. Small deviations are expected as thermodynamic correlations and numerical solvers evolve; a deviation is a review trigger, not something to hide with a hard-coded correction.


In [ ]:
Report = JClass('neqsim.process.util.report.Report')
process = build_process(); process.run()
report = json.loads(str(Report(process).generateJsonReport()))

def value(path):
    item = report
    for key in path:
        item = item[key]
    return float(item['value'] if isinstance(item, dict) and 'value' in item else item)

compressor_tags = ['23-KA-03', '23-KA-02', '23-KA-01', '27-KA-01']
power_values = {tag: float(report[tag]['power']) for tag in compressor_tags}
pump_power_kw = float(report['21-PA-01']['power']) / 1000.0
current = {
    'Gas export rate [kmol/h]': value(['export gas', 'conditions', 'overall', 'molar flow']) / 1000.0,
    'Oil export rate [kmol/h]': value(['export oil', 'conditions', 'overall', 'molar flow']) / 1000.0,
    'Oil molecular weight [g/mol]': value(['export oil', 'properties', 'overall', 'molar mass']) * 1000.0,
    'Gas molecular weight [g/mol]': value(['export gas', 'properties', 'overall', 'molar mass']) * 1000.0,
    'Oil RVP [psia]': value(['export oil', 'properties', 'oil', 'RVP']) * 14.5038,
    'Oil TVP [bara]': value(['export oil', 'properties', 'oil', 'TVP']),
    'Gas GCV 15/15 [MJ/Sm3]': value(['export gas', 'properties', 'gas', 'GCV (15/15)']),
    'Gas Wobbe 15/15 [MJ/Sm3]': value(['export gas', 'properties', 'gas', 'WI (15/15)']),
    'Total rotating power [kW]': sum(power_values.values()) + pump_power_kw,
}
benchmark = pd.DataFrame([
    {'result': key, 'published': expected, 'current': current[key],
     'deviation [%]': 100.0 * (current[key] - expected) / expected}
    for key, expected in PUBLISHED.items()
])
display(benchmark.style.format({'published': '{:.4f}', 'current': '{:.4f}', 'deviation [%]': '{:+.2f}'}))
assert benchmark['deviation [%]'].abs().max() < 10.0, 'Review benchmark regression above 10%'


In [ ]:
exchanger_tags = ['20-HA-01', '20-HA-02', '20-HA-03', '21-HA-01', '23-HA-01',
                  '23-HA-02', '23-HA-03', '24-HA-01', '25-HA-01', '25-HA-02', '27-HA-01']
def report_duty_kw(tag):
    item = report[tag]
    raw = item.get('data', {}).get('duty', item.get('duty', 0.0))
    return float(raw['value'] if isinstance(raw, dict) else raw) / 1000.0

duty_rows = [{'tag': tag, 'duty [kW]': report_duty_kw(tag)} for tag in exchanger_tags]
power_rows = [{'tag': tag, 'power [kW]': power_values[tag],
               'discharge temperature [C]': float(report[tag]['dischargeTemperature'])}
              for tag in compressor_tags]
power_rows.append({'tag': '21-PA-01', 'power [kW]': pump_power_kw,
                   'discharge temperature [C]': float(report['21-PA-01']['dischargeTemperature'])})
display(pd.DataFrame(duty_rows)); display(pd.DataFrame(power_rows))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar([row['tag'] for row in duty_rows], [row['duty [kW]'] for row in duty_rows], color='#2878B5')
axes[0].set(ylabel='Duty [kW]', xlabel='Heat exchanger', title='Normal-case heating and cooling duties')
axes[0].tick_params(axis='x', rotation=55); axes[0].grid(axis='y', alpha=0.3)
axes[1].bar([row['tag'] for row in power_rows], [row['power [kW]'] for row in power_rows], color='#D95319')
axes[1].set(ylabel='Shaft power [kW]', xlabel='Rotating equipment', title='Normal-case rotating power')
axes[1].tick_params(axis='x', rotation=45); axes[1].grid(axis='y', alpha=0.3)
fig.tight_layout(); plt.show()


**Figure discussion — duties and power.** Observation: the first export-compression stage and oil export pump are major electrical consumers, while the export aftercooler and oil cooler dominate cooling demand in the published base case. Mechanism: the gas is raised from first-stage pressure to 90 barg and then to 188.6 barg, and the stabilized oil is raised from LP-separator pressure to 60 barg. Engineering implication: compressor drivers, cooling-medium systems, electrical distribution, and oil-pump selection must use the multi-case envelope rather than the normal case. Recommendation: preserve at least 10% screening margin, then replace assumed efficiencies and heat-transfer coefficients with vendor and utility data.


## 3. Operating and accidental case matrices

Seven steady cases are executable process/design cases. The accidental cases are evaluated by the governed safety-scenario calculation because a label alone is not a dynamic compressor-trip, fire, or depressurization model.


In [ ]:
STEADY_CASES = [
    ('minimum-turndown', 'MINIMUM_TURNDOWN', 0.60, 50.0, 31.5, 'Minimum stable production'),
    ('normal', 'NORMAL', 1.00, 60.0, 31.5, 'Published comparison case'),
    ('maximum-production', 'MAXIMUM_PRODUCTION', 1.20, 60.0, 31.5, 'Capacity and driver governing candidate'),
    ('cold-feed', 'AMBIENT_EXTREME', 1.00, 40.0, 31.5, 'Cold inlet and heating-duty check'),
    ('hot-feed', 'AMBIENT_EXTREME', 1.00, 75.0, 31.5, 'Hot inlet and cooling-duty check'),
    ('high-inlet-pressure', 'CUSTOM', 1.00, 60.0, 35.0, 'HP separator pressure envelope'),
    ('low-inlet-pressure', 'CUSTOM', 1.00, 60.0, 28.0, 'Recompression-power envelope'),
]
ACCIDENTAL_CASES = [
    ('blocked-gas-export', 'BLOCKED_OUTLET', '27-KA-01 / export piping'),
    ('cooling-failure', 'UTILITY_OR_COOLING_FAILURE', '23/24/25/27 coolers'),
    ('compressor-trip-settle-out', 'COMPRESSOR_TRIP_SETTLE_OUT', '23/27 compression trains'),
    ('fire-exposure', 'FIRE_EXPOSURE', 'separators and scrubbers'),
    ('simultaneous-blowdown', 'SIMULTANEOUS_BLOWDOWN', 'common flare interface'),
]
display(pd.DataFrame(STEADY_CASES, columns=['case', 'type', 'rate factor', 'feed C', 'HP sep barg', 'purpose']))
display(pd.DataFrame(ACCIDENTAL_CASES, columns=['scenario', 'type', 'scope']))


In [ ]:
NorsokBuilder = JClass('neqsim.process.engineering.NorsokOffshoreEngineeringBuilder')
DesignBuilder = JClass('neqsim.process.engineering.ProcessToEngineeringDesignBuilder')
DesignCase = JClass('neqsim.process.engineering.designcase.EngineeringDesignCase')
DesignCaseInput = JClass('neqsim.process.engineering.designcase.EngineeringDesignCase$Input')
Configurator = JClass('neqsim.process.engineering.designcase.EngineeringDesignCase$Configurator')
EngineeringMetric = JClass('neqsim.process.engineering.designcase.EngineeringMetric')

project = NorsokBuilder.from_('Complete offshore process engineering study', process) \
    .projectId('OFFSHORE-PROCESS-STUDY').registerProposedInstruments(True).build().setRevision('A')
case_callbacks = []
for case_id, case_type, rate_factor, feed_temperature, hp_pressure, purpose in STEADY_CASES:
    params = dict(BASE); params['feed_rate_kmol_h'] *= rate_factor
    params['feed_temperature_C'] = feed_temperature; params['Psep1_barg'] = hp_pressure
    def configure(case_process, params=params):
        apply_case(case_process, params)
    proxy = JProxy(Configurator, dict={'configure': configure}); case_callbacks.append(proxy)
    case = DesignCase(case_id, purpose, getattr(DesignCase.Type, case_type), proxy)
    case.setCaseGroup('OPERATING-ENVELOPE').setApprovalStatus('REVIEW_REQUIRED')
    case.addInput(DesignCaseInput('feedRate', params['feed_rate_kmol_h'], 'kmol/hr', 'COLAB-COMPARISON-BASIS'))
    case.addInput(DesignCaseInput('feedTemperature', feed_temperature, 'C', 'SCREENING-ENVELOPE'))
    case.addInput(DesignCaseInput('firstStagePressure', hp_pressure, 'barg', 'SCREENING-ENVELOPE'))
    project.addDesignCase(case)

drivers = jdouble([500, 1000, 2000, 3000, 5000, 7500, 10000, 12500, 15000, 20000])
areas = jdouble([50, 100, 200, 400, 800, 1200, 1800, 2500, 4000])
volumes = jdouble([5, 10, 20, 40, 80, 120, 200, 300, 500])
cvs = jdouble([50, 100, 250, 500, 800, 1200, 2000])
builder = DesignBuilder.on(project).separatorBasis(750.0, 0.107, 180.0) \
    .exportLineLimits(20.0, 5.0).compressorDrivers(0.10, drivers)
builder.addInletCompressionExportSlice('20-VA-01', '23-KA-01', 'EXPORT-GAS-LINE', 'VLV-100', '20-PIT-001')
for tag in ['20-VA-02', '20-VA-03', '23-VG-01', '23-VG-02', '23-VG-03', '24-VG-01', '25-VG-01']:
    builder.addInventoryDesign(tag, 180.0, 0.70, volumes)
for tag in exchanger_tags:
    builder.addHeatExchangerDesign(tag, 500.0, 25.0, 0.15, areas)
builder.addPumpDesign('21-PA-01', 0.10, 3.0, drivers)
builder.addPumpDesign('23-PA-01', 0.10, 1.0, drivers)
builder.addControlValveDesign('VLV-102', 70.0, 85.0, cvs)
for tag in ['23-KA-02', '23-KA-03', '27-KA-01']:
    builder.addRatedCapacity(tag, EngineeringMetric.compressorPower(tag),
        'driverRatedPower', 'kW', 0.10, drivers)
print('Executable cases:', project.getExecutableDesignCases().size())
print('Engineering design modules:', project.getEngineeringDesignModules().size())


## 4. Closed process-to-engineering design loop

Each case runs on an isolated process copy. Discrete selections and continuous dimensions are applied to a separate designed process until physical variables, process values, and constraints stabilize. Depending on hardware, the full seven-case process may take several minutes.


In [ ]:
Simulator = JClass('neqsim.process.engineering.ProcessToEngineeringSimulator')
simulation = Simulator.run(project, min(4, os.cpu_count() or 1))
loop = simulation.getEngineeringDesignLoopResult()
design_rows = []
for entry in loop.getState().getValues().entrySet():
    item = entry.getValue().toMap()
    design_rows.append({'variable': str(entry.getKey()), 'value': item.get('value'),
                        'unit': str(item.get('unit')), 'governing case': str(item.get('governingCaseId')),
                        'source module': str(item.get('sourceModule'))})
design_table = pd.DataFrame(design_rows).sort_values('variable').reset_index(drop=True)
print('Converged:', loop.isConverged(), 'iterations:', loop.getIterations().size(), loop.getTerminationReason())
display(design_table)
convergence_rows = []
for iteration in loop.getIterations():
    convergence = iteration.getConvergenceReport()
    convergence_rows.append({
        'iteration': iteration.getNumber(),
        'applied updates': iteration.getAppliedUpdateCount(),
        'maximum design change': iteration.getMaximumRelativeChange(),
        'maximum process change': convergence.getMaximumProcessValueRelativeChange(),
        'constraints satisfied': convergence.getSatisfiedConstraintCount(),
        'constraints evaluated': convergence.getConstraintCount(),
        'process stable': convergence.areProcessValuesStable(),
        'cases successful': convergence.areCaseRunsSuccessful(),
    })
convergence_table = pd.DataFrame(convergence_rows)
display(convergence_table)
print(convergence_table.to_string(index=False))
constraint_table = pd.DataFrame([
    {str(key): value for key, value in constraint.toMap().entrySet()}
    for constraint in loop.getIterations().get(loop.getIterations().size() - 1).getConstraintResults()
])
display(constraint_table)
print('Unsatisfied constraints:')
print(constraint_table.loc[~constraint_table['satisfied'].astype(bool)].to_string(index=False))
final_case_results = list(loop.getIterations().get(loop.getIterations().size() - 1)
                          .getCaseReport().getEnvelope().getCaseResults())
case_status_table = pd.DataFrame([
    {'case': str(result.getDesignCase().getId()), 'status': str(result.getStatus()),
     'converged': result.isConverged(), 'message': str(result.getMessage())}
    for result in final_case_results
])
display(case_status_table)
print(case_status_table.to_string(index=False))
metric_failure_rows = []
for result in final_case_results:
    for entry in result.getMetricResults().entrySet():
        metric_result = entry.getValue()
        if str(metric_result.getStatus()) == 'FAILED':
            detail = {str(key): value for key, value in metric_result.toMap().entrySet()}
            metric_failure_rows.append({'case': str(result.getDesignCase().getId()), **detail})
metric_failure_table = pd.DataFrame(metric_failure_rows)
print('Metric failures:')
print(metric_failure_table.to_string(index=False) if len(metric_failure_table) else 'none')
assert loop.isConverged()
assert not simulation.getCaseRunReport().getEnvelope().hasCaseFailures()


In [ ]:
iterations = list(loop.getIterations())
x = [item.getNumber() for item in iterations]
updates = [item.getAppliedUpdateCount() for item in iterations]
physical = [min(item.getMaximumRelativeChange(), 10.0) for item in iterations]
process_change = [min(item.getConvergenceReport().getMaximumProcessValueRelativeChange(), 10.0) for item in iterations]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.0))
axes[0].bar(x, updates, color='#2878B5'); axes[0].set(xlabel='Iteration', ylabel='Applied updates', title='Design updates')
axes[1].semilogy(x, physical, marker='o', color='#D95319'); axes[1].set(xlabel='Iteration', ylabel='Maximum relative change [-]', title='Physical convergence')
axes[2].semilogy(x, process_change, marker='s', color='#2E8B57'); axes[2].set(xlabel='Iteration', ylabel='Maximum relative change [-]', title='Process convergence')
for ax in axes: ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()


**Figure discussion — convergence.** Observation: the first pass selects preliminary geometry and ratings; later passes re-run the connected process with those selections until no material update remains. Mechanism: line diameter, separator size, heat-transfer area, valve Cv, and driver ratings alter pressure drop or equipment state and can move the governing case. Engineering implication: a one-pass datasheet export can be internally inconsistent. Recommendation: review every unsatisfied constraint and governing-case change, and retain the iteration evidence in the package.


In [ ]:
case_results = list(simulation.getCaseRunReport().getEnvelope().getCaseResults())
envelope_rows = []
for item in case_results:
    values = item.getValues()
    rotating = sum(float(values.get(tag + '.power') or 0.0) for tag in compressor_tags)
    envelope_rows.append({'case': str(item.getDesignCase().getId()), 'compressor power [kW]': rotating,
                          'successful': str(item.getStatus()) == 'CALCULATED'})
envelope_table = pd.DataFrame(envelope_rows)
display(envelope_table)
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(envelope_table['case'], envelope_table['compressor power [kW]'], color='#6F4E7C')
ax.set(xlabel='Controlled steady design case', ylabel='Total compressor shaft power [kW]',
       title='Full-process compressor-power envelope')
ax.tick_params(axis='x', rotation=35); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); plt.show()


**Figure discussion — case envelope.** Observation: the maximum-rate or low-inlet-pressure case normally governs total compression power, while temperature extremes govern different exchanger duties. Mechanism: recompression ratio and gas volumetric flow change simultaneously across the multi-stage train. Engineering implication: no single “design case” controls every equipment item. Recommendation: retain the governing case on each selected variable and verify driver, exchanger, line, valve, and vessel margins independently.


## 5. Discipline calculations for every equipment family in this process

The design loop gives connected, multi-case selections. The following typed calculations expose the underlying preliminary process/mechanical results for all separators and scrubbers, compressors, pumps, and exchangers. Screening assumptions such as surrogate surge/choke flow, U-value, NPSH required, corrosion allowance, and MDMT are deliberately visible.


In [ ]:
Context = JClass('neqsim.process.engineering.calculation.EngineeringCalculationContext')
context = Context.builder().designCaseId('maximum-production').build()
EquipmentInput = JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Input')
SepCalc = JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Separator')
CompCalc = JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Compressor')
PumpCalc = JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Pump')
HxCalc = JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$HeatExchanger')

def map_row(tag, family, result):
    row = {'tag': tag, 'family': family, 'status': str(result.getStatus())}
    if result.getValue() is not None:
        row.update({str(k): v for k, v in result.getValue().toMap().entrySet()})
    return row

separator_tags = ['20-VA-01', '20-VA-02', '20-VA-03', '23-VG-01', '23-VG-02',
                  '23-VG-03', '24-VG-01', '25-VG-01']
equipment_rows = []
for tag in separator_tags:
    unit = process.getUnit(tag); gas = unit.getGasOutStream()
    liquid = unit.getOilOutStream() if tag.startswith('20-VA') else unit.getLiquidOutStream()
    gas.getFluid().initProperties(); liquid.getFluid().initProperties()
    inputs = EquipmentInput.builder(tag, 'maximum-production', 'COLAB-COMPARISON-BASIS') \
        .value('gasFlowM3s', max(gas.getFlowRate('m3/sec'), 1.0e-8)) \
        .value('liquidFlowM3s', max(liquid.getFlowRate('m3/sec'), 1.0e-8)) \
        .value('gasDensityKgM3', max(gas.getFluid().getDensity('kg/m3'), 0.1)) \
        .value('liquidDensityKgM3', max(liquid.getFluid().getDensity('kg/m3'), 100.0)).build()
    equipment_rows.append(map_row(tag, 'separator', SepCalc().calculate(inputs, context)))

for tag in compressor_tags:
    unit = process.getUnit(tag); flow = max(unit.getInletStream().getFlowRate('m3/sec'), 1.0e-8)
    inputs = EquipmentInput.builder(tag, 'maximum-production', 'SCREENING-MAP-BASIS') \
        .value('operatingFlowM3s', flow).value('surgeFlowM3s', 0.70 * flow) \
        .value('chokeFlowM3s', 1.40 * flow).value('shaftPowerKw', unit.getPower('kW')).build()
    equipment_rows.append(map_row(tag, 'compressor', CompCalc().calculate(inputs, context)))

for tag in ['21-PA-01', '23-PA-01']:
    unit = process.getUnit(tag); inlet = unit.getInletStream(); inlet.getFluid().initProperties()
    rho = max(inlet.getFluid().getDensity('kg/m3'), 100.0)
    head = max((unit.getOutletStream().getPressure('bara') - inlet.getPressure('bara')) * 1.0e5 / rho / 9.80665, 1.0)
    inputs = EquipmentInput.builder(tag, 'maximum-production', 'SCREENING-PUMP-BASIS') \
        .value('flowM3s', max(inlet.getFlowRate('m3/sec'), 1.0e-8)).value('headM', head) \
        .value('npshaM', 8.0).value('npshrM', 4.0).value('shaftPowerKw', unit.getPower() / 1000.0).build()
    equipment_rows.append(map_row(tag, 'pump', PumpCalc().calculate(inputs, context)))

for tag in exchanger_tags:
    inputs = EquipmentInput.builder(tag, 'maximum-production', 'SCREENING-THERMAL-BASIS') \
        .value('dutyKw', max(abs(report_duty_kw(tag)), 1.0e-6)).value('overallU_WPerM2K', 500.0) \
        .value('correctedLmtdK', 25.0).build()
    equipment_rows.append(map_row(tag, 'heat exchanger', HxCalc().calculate(inputs, context)))
equipment_design_table = pd.DataFrame(equipment_rows)
display(equipment_design_table)


In [ ]:
Piping = 'neqsim.process.engineering.piping.PipingNetworkDesignCalculation'
Candidate = JClass(Piping + '$Candidate'); HydraulicCase = JClass(Piping + '$Case')
Segment = JClass(Piping + '$Segment'); NetworkInput = JClass(Piping + '$Input')
ArrayList = JClass('java.util.ArrayList'); LinkedHashMap = JClass('java.util.LinkedHashMap')
candidates = ArrayList()
for name, diameter in [('6in', 0.154), ('8in', 0.203), ('10in', 0.254), ('12in', 0.303), ('16in', 0.394), ('20in', 0.489)]:
    candidates.add(Candidate(name, '40', diameter, 250.0))
gas_flow = max(process.getUnit('EXPORT-GAS-LINE').getInletStream().getFlowRate('m3/sec'), 1.0e-6)
oil_flow = max(process.getUnit('EXPORT-OIL-LINE').getInletStream().getFlowRate('m3/sec'), 1.0e-6)
gas_segment = Segment('EXPORT-GAS-LINE', True, False, 2000.0, 0.0, 10.0, 1.0, 0.0) \
    .addCase(HydraulicCase('maximum-production', 1.20 * gas_flow, 0.10, 0.20, 190.0, 40.0))
oil_segment = Segment('EXPORT-OIL-LINE', True, False, 2000.0, 0.0, 5.0, 1.0, 0.0) \
    .addCase(HydraulicCase('maximum-production', 1.20 * oil_flow, 0.10, 0.20, 61.0, 49.0))
segments = ArrayList(); segments.add(gas_segment); segments.add(oil_segment)
demands = LinkedHashMap(); demands.put('gas export', 1.20 * gas_flow); demands.put('oil export', 1.20 * oil_flow)
rules = JClass('neqsim.process.engineering.piping.PipingRulePack').norsokP0022023Ac2024()
network = JClass(Piping)().calculate(NetworkInput('terminal-export-network', rules, candidates, segments, demands), context)
network_table = pd.DataFrame([{'segment': str(k), **{str(a): b for a, b in v.entrySet()}}
                              for k, v in network.getValue().toMap().get('selections').entrySet()])
display(network_table)


In [ ]:
ValveInput = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$ValveInput')
ValveCalc = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$Valve')
Failure = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$FailurePosition')
valve_rows = []
for tag, selected_cv in [('VLV-100', 800.0), ('VLV-102', 600.0)]:
    valve = process.getUnit(tag); inlet = valve.getInletStream(); outlet = valve.getOutletStream()
    inlet.getFluid().initProperties()
    result = ValveCalc().calculate(ValveInput(tag, 'maximum-production', 0.8 * selected_cv, selected_cv,
        70.0, inlet.getPressure('bara'), outlet.getPressure('bara'), 1.0, 50.0,
        max(inlet.getFluid().getDensity('kg/m3'), 1.0), inlet.getFlowRate('kg/sec'),
        8.0, 10.0, Failure.HAZOP_INPUT_REQUIRED), context)
    valve_rows.append(map_row(tag, 'valve', result))

InstrumentInput = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$InstrumentInput')
InstrumentCalc = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$Instrument')
instrument_rows = []
for tag, low, high, upper in [('20-PIT-001', 28.0, 35.5, 40.0), ('20-TIT-001', 40.0, 75.0, 100.0),
                              ('23-PIT-001', 8.0, 35.0, 40.0), ('25-TIT-001', 5.0, 20.0, 50.0),
                              ('27-PIT-001', 180.0, 200.0, 250.0), ('21-PIT-001', 55.0, 65.0, 100.0)]:
    result = InstrumentCalc().calculate(InstrumentInput(tag, 'operating-envelope', low, high, 0.0, upper,
        0.005, 1.0, 0.5, 8.0, 15.0, 2.0, 30.0), context)
    instrument_rows.append(map_row(tag, 'instrument', result))
display(pd.DataFrame(valve_rows)); display(pd.DataFrame(instrument_rows))


In [ ]:
Safety = 'neqsim.process.engineering.safety.SafetyScenarioEngineCalculation'
Scenario = JClass(Safety + '$Scenario'); ScenarioType = JClass(Safety + '$Type')
Credibility = JClass(Safety + '$Credibility'); FluidModel = JClass(Safety + '$FluidModel')
scenarios = ArrayList()
scenario_inputs = [
    ('blocked-gas-export', '27-KA-01', ScenarioType.BLOCKED_OUTLET, FluidModel.GAS, 'GAS-TRAIN', 30.0, 200.0, 1.287, -20.0),
    ('cooling-failure', '24-VG-01', ScenarioType.UTILITY_OR_COOLING_FAILURE, FluidModel.GAS, 'GAS-TRAIN', 20.0, 100.0, 0.785, -20.0),
    ('compressor-trip', '23-KA-01', ScenarioType.COMPRESSOR_TRIP_SETTLE_OUT, FluidModel.GAS, 'GAS-TRAIN', 15.0, 100.0, 0.503, -35.0),
    ('fire-hp-separator', '20-VA-01', ScenarioType.FIRE_EXPOSURE, FluidModel.TWO_PHASE, 'FIRE-ZONE-20', 25.0, 40.0, 1.287, -20.0),
    ('blocked-oil-export', '21-PA-01', ScenarioType.BLOCKED_OUTLET, FluidModel.LIQUID, 'LIQUID-TRAIN', 12.0, 70.0, 0.503, 0.0),
    ('simultaneous-blowdown', '24-VG-01', ScenarioType.SIMULTANEOUS_BLOWDOWN, FluidModel.GAS, 'BLOWDOWN-ALL', 45.0, 100.0, 1.838, -46.0),
]
for sid, tag, stype, fluid_model, group, mass, pressure, area, minimum_temperature in scenario_inputs:
    scenarios.add(Scenario(sid, tag, stype, Credibility.CREDIBLE, fluid_model, group,
        'SCREENING-CREDIBILITY-ASSUMPTION-NOT-HAZOP-APPROVED', mass, pressure, area,
        2.0, 5.0, 1.0, minimum_temperature))
orifices = jdouble([0.110, 0.196, 0.307, 0.503, 0.785, 1.287, 1.838, 2.853, 4.34])
safety = JClass(Safety)().calculate(JClass(Safety + '$Input')('full-process-safety-screen', scenarios,
    orifices, 3.0, 10.0, 120.0, -46.0), context)
safety_map = safety.getValue().toMap()
safety_table = pd.DataFrame([{str(k): v for k, v in row.entrySet()}
                             for row in safety_map.get('scenarios').values()])
display(safety_table)
display(pd.DataFrame([{'concurrency group': str(k), 'load [kg/s]': v}
                      for k, v in safety_map.get('concurrencyGroupLoadsKgS').entrySet()]))


In [ ]:
MaterialInput = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$MaterialInput')
MaterialCalc = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$MaterialSelection')
MechanicalInput = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$MechanicalInput')
MechanicalCalc = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$PreliminaryMechanical')
materials_rows, mechanical_rows = [], []
for tag in separator_tags:
    operating_pressure = process.getUnit(tag).getInletStream().getPressure('bara')
    material = MaterialCalc().calculate(MaterialInput(tag, 'operating-envelope', 0.0159, 0.0, 100.0,
        True, False, 70.0, 100.0, -46.0, 1.10 * operating_pressure, 3.0, 25.0,
        'marine offshore external environment'), context)
    materials_rows.append(map_row(tag, 'materials', material))
    design = design_table[design_table['variable'].str.startswith(tag + '.')]
    diameter = float(design.loc[design['variable'].str.contains('insideDiameter'), 'value'].iloc[0]) if design['variable'].str.contains('insideDiameter').any() else 2.0
    length = float(design.loc[design['variable'].str.contains('tangentLength'), 'value'].iloc[0]) if design['variable'].str.contains('tangentLength').any() else 6.0
    mechanical = MechanicalCalc().calculate(MechanicalInput(tag, 'operating-envelope', operating_pressure,
        1.10 * operating_pressure, diameter, length, 138.0, 0.85, 3.0, -46.0, 0.10), context)
    mechanical_rows.append(map_row(tag, 'mechanical', mechanical))
display(pd.DataFrame(materials_rows)); display(pd.DataFrame(mechanical_rows))


## 6. Coordinated deliverables and readiness boundary

The compiler uses the same designed process, case envelope, graph identities, and calculation evidence to produce mutually consistent discipline handoffs. Production readiness is assessed fail-closed: missing independent benchmarks, vendor guarantees, HAZOP/LOPA/SRS approvals, distributed transients, detailed mechanical integrity, and construction-authority evidence remain named gaps.


In [ ]:
Compiler = JClass('neqsim.process.engineering.deliverables.EngineeringDeliverableCompiler')
Readiness = JClass('neqsim.process.engineering.production.EngineeringProductionReadinessAssessment')
ReadinessBasis = JClass('neqsim.process.engineering.production.EngineeringProductionReadinessBasis')
Paths = JClass('java.nio.file.Paths')
package_dir = ROOT / 'build' / 'complete-offshore-process-engineering-study'
shutil.rmtree(package_dir, ignore_errors=True); package_dir.mkdir(parents=True)
compiled = Compiler.compile(project, Paths.get(str(package_dir)))
readiness = Readiness.assess(project, ReadinessBasis())
readiness_map = readiness.toMap()
required_artifacts = [
    'engineering-model.json', 'engineering-design-case-matrix.json', 'design-case-envelope.json',
    'equipment-register.json', 'line-register.json', 'instrument-register.json', 'valve-list.json',
    'io-list.json', 'alarm-trip-schedule.json', 'psv-datasheets.json', 'utility-summary.json',
    'materials-selection-report.json', 'engineering-calculation-dag.json', 'plant.dexpi.xml',
    'plant.dexpi20.xml', 'engineering-dexpi-roundtrip-report.json',
    'unresolved-engineering-actions.json', 'engineering-validation-report.json',
]
artifact_table = pd.DataFrame([{'artifact': name, 'exists': (package_dir / name).is_file()}
                               for name in required_artifacts])
gate_table = pd.DataFrame([{'gate': str(name), 'passed': item.get('passed'),
                            'required action': str(item.get('requiredAction'))}
                           for name, item in readiness_map.get('gates').entrySet()])
display(artifact_table); display(gate_table)
print('Package structurally valid:', compiled.getValidationReport().isValid())
print('Maturity level:', readiness.getLevel())
print('fitnessForConstruction =', readiness_map.get('fitnessForConstruction'))
print('finalEngineeringApprovalGranted =', readiness_map.get('finalEngineeringApprovalGranted'))
assert artifact_table['exists'].all()
assert not bool(readiness_map.get('fitnessForConstruction'))


## 7. Engineering conclusions

The normal case reproduces the published process benchmark: approximately 5,112 kmol/h export gas, 2,753 kmol/h export oil, 10.25 psia oil RVP, 45.68 MJ/Sm3 gas GCV, 53.49 MJ/Sm3 Wobbe index, and 10.38 MW total compressor-plus-oil-pump power. The connected design loop identifies the governing case separately for separator geometry, inventory, exchanger area, line diameter, valve Cv, and driver rating. Discipline tables then expose preliminary separator, compressor, pump, exchanger, piping, valve/instrument, relief/blowdown/flare, materials, and mechanical results for the full train.

The package is a reproducible concept/pre-FEED engineering basis, not a construction design. Before project use, replace every screening assumption with controlled inputs and close the generated actions for fluid validation, HAZOP/LOPA/SRS, compressor and pump maps, exchanger vendor thermal design, two-phase relief method, flare-network/consequence analysis, piping stress and transients, valve noise/cavitation and actuator sizing, instrument installation/thermowells, corrosion/materials approval, pressure-vessel code design, layout/maintainability, and accountable release.
